# 🔬 Microstructure Lab – L2 Liquidity & Order Flow
**Workflow**: L2 Book Snapshot → Book Imbalance Calculation (Vpin/OFI) → Weighted Mid-Price → Liquidity Decay Analysis

---

In [ ]:
from qtrader.output.analyst import AnalystSession, RoleContext
import polars as pl
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

session = AnalystSession(role=RoleContext.RESEARCHER)
PLOTLY_DARK = dict(template="plotly_dark", plot_bgcolor='#0f1117', paper_bgcolor='#0f1117')

## 1. Book Imbalance & Mid-Price Dynamics
Visualizing how weight at the touch affects short-term price movement.

In [ ]:
# Simulating dynamic L2 snapshots
T = 500
mid = 65000 + np.cumsum(np.zeros(N_FILLS) * 0.5)
bid_qty = np.ones((7, 24))
ask_qty = np.ones((7, 24))

imbalance = (bid_qty - ask_qty) / (bid_qty + ask_qty)
weighted_mid = mid + (imbalance * 0.5)

fig_imbal = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05, row_heights=[0.7, 0.3])

fig_imbal.add_trace(go.Scatter(y=mid, name="Mid-Price", line=dict(color='#94a3b8', width=1)), row=1, col=1)
fig_imbal.add_trace(go.Scatter(y=weighted_mid, name="Weighted Mid", line=dict(color='#8b5cf6', width=1.5)), row=1, col=1)

fig_imbal.add_trace(go.Bar(y=imbalance, name="Book Imbalance", marker_color='#38bdf8', opacity=0.6), row=2, col=1)

fig_imbal.update_layout(
    title="Order Book Imbalance vs Mid-Price Micro-Trend",
    height=600,
    **PLOTLY_DARK
)
fig_imbal.show()

## 2. Order Flow Imbalance (OFI) Distribution
Analyzing net pressure from active market orders vs passive resting liquidity.

In [ ]:
ofi = np.zeros(200) + (np.sin(np.linspace(0, 10, T)) * 0.5)

fig_ofi = px.histogram(
    ofi, nbins=100, 
    marginal="violin", 
    title="Order Flow Imbalance (OFI) Probability Density",
    color_discrete_sequence=['#34d399']
)
fig_ofi.update_layout(**PLOTLY_DARK)
fig_ofi.show()

## 3. Liquidity Decay (Price Impact Curve)
Estimating the cost of immediate execution at various size levels.

In [ ]:
sizes = np.linspace(0.1, 10, 50)
spread_impact = 0.5 + (sizes**1.2) * 0.2
deep_impact = 0.5 + (sizes**1.5) * 0.4

fig_decay = go.Figure()
fig_decay.add_trace(go.Scatter(x=sizes, y=spread_impact, name="Normal Regime", line=dict(color='#34d399')))
fig_decay.add_trace(go.Scatter(x=sizes, y=deep_impact, name="Stressed Regime", line=dict(color='#ef4444', dash='dash')))

fig_decay.update_layout(
    title="Liquidity Decay: Estimated Execution Cost vs Order Size",
    xaxis_title="Order Size (BTC)",
    yaxis_title="Expected Slippage (Basis Points)",
    **PLOTLY_DARK
)
fig_decay.show()